## Task 4: Base Model Implementation


In [1]:
!pip install -U transformers datasets evaluate accelerate scikit-learn
!pip install -q nltk wordcloud spacy umap-learn
!pip install matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.7 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling tran

### Load and process the PCL Dataset

In [2]:
! mkdir -p data

# Download train/dev split file (practice split)
! wget -O data/practice_splits.zip \https://github.com/Perez-AlmendrosC/dontpatronizeme/archive/refs/heads/master.zip

# Download test set (raw TSV)
! wget -O data/task4_test.tsv \https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/TEST/task4_test.tsv

# Download dataset
! wget -O data/dontpatronizeme_pcl.tsv \https://raw.githubusercontent.com/CRLala/NLPLabs-2024/refs/heads/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv

# Unzip the dataset
!unzip data/practice_splits.zip -d data/

--2026-03-03 23:59:16--  https://github.com/Perez-AlmendrosC/dontpatronizeme/archive/refs/heads/master.zip
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://codeload.github.com/Perez-AlmendrosC/dontpatronizeme/zip/refs/heads/master [following]
--2026-03-03 23:59:17--  https://codeload.github.com/Perez-AlmendrosC/dontpatronizeme/zip/refs/heads/master
Resolving codeload.github.com (codeload.github.com)... 20.205.243.165
Connecting to codeload.github.com (codeload.github.com)|20.205.243.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘data/practice_splits.zip’

data/practice_split     [ <=>                ] 664.12K  --.-KB/s    in 0.1s    

2026-03-03 23:59:17 (6.52 MB/s) - ‘data/practice_splits.zip’ saved [680064]

--2026-03-03 23:59:17--  https://raw.githubusercontent.com/Perez

In [3]:
import pandas as pd

# Load the full PCL dataset
full_df = pd.read_csv(
    "data/dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=3, # disclaimer is lines 1–3, data starts at line 4
    header=None
)

# Assign column names
full_df.columns = ["par_id", "art_id", "keyword", "country_code", "text", "label"]

# Convert graded label (0–4) -> binary (0/1) using {0,1}=NoPCL and {2,3,4}=PCL
full_df["label"] = full_df["label"].astype(int)
full_df["label_bin"] = (full_df["label"] >= 2).astype(int)

# Load split IDs
train_ids = pd.read_csv("data/dontpatronizeme-master/semeval-2022/practice splits/train_semeval_parids-labels.csv")
dev_ids   = pd.read_csv("data/dontpatronizeme-master/semeval-2022/practice splits/dev_semeval_parids-labels.csv")

# Ensure types match
full_df["par_id"] = full_df["par_id"].astype(int)
train_ids["par_id"] = train_ids["par_id"].astype(int)
dev_ids["par_id"] = dev_ids["par_id"].astype(int)

# Merge to get actual train/dev dataframes
train_df = full_df.merge(train_ids[["par_id"]], on="par_id", how="inner")
dev_df   = full_df.merge(dev_ids[["par_id"]], on="par_id", how="inner")

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("\nTrain label distribution:\n", train_df["label_bin"].value_counts())

train_df["text"] = train_df["text"].fillna("").astype(str)
dev_df["text"]   = dev_df["text"].fillna("").astype(str)

train_df["label_bin"] = train_df["label_bin"].astype(int)
dev_df["label_bin"]   = dev_df["label_bin"].astype(int)

Train shape: (8375, 7)
Dev shape: (2094, 7)

Train label distribution:
 label_bin
0    7581
1     794
Name: count, dtype: int64


In [4]:
# Build HF datasets from the DataFrames
from datasets import Dataset

train_hf = Dataset.from_pandas(train_df[["text", "label_bin"]].copy())
dev_hf   = Dataset.from_pandas(dev_df[["text", "label_bin"]].copy())

train_hf = train_hf.rename_column("label_bin", "labels")
dev_hf   = dev_hf.rename_column("label_bin", "labels")

In [5]:
from transformers import AutoTokenizer

checkpoint = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=512)

train_hf = train_hf.map(tokenize_fn, batched=True)
dev_hf   = dev_hf.map(tokenize_fn, batched=True)

train_hf.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
dev_hf.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/8375 [00:00<?, ? examples/s]

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
# Metrics: F1-score, precision, recall
import evaluate
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {"precision": precision, "recall": recall, "f1": f1}

In [8]:
# Training
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=dev_hf,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.287471,0.263626,0.437751,0.547739,0.486607
2,0.218229,0.248663,0.579618,0.457286,0.511236
3,0.153453,0.323742,0.577778,0.391960,0.467066


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

{'eval_loss': 0.24882499873638153,
 'eval_precision': 0.5796178343949044,
 'eval_recall': 0.457286432160804,
 'eval_f1': 0.5112359550561798,
 'eval_runtime': 60.5604,
 'eval_samples_per_second': 34.577,
 'eval_steps_per_second': 2.163,
 'epoch': 3.0}